# Transferencia

**Capítulo 4 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_computer-vision/fine-tuning.ipynb` · [Lección original](https://d2l.ai/chapter_computer-vision/fine-tuning.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Fine-tuning
<a id="sec_fine_tuning"></a>

En capítulos anteriores, discutimos cómo entrenar modelos en el conjunto de datos de entrenamiento de Fashion-MNIST con sólo 60000 imágenes. También describimos ImageNet, el conjunto de datos de imagen a gran escala más utilizado en el mundo académico, que tiene más de 10 millones de imágenes y 1000 objetos. Sin embargo, el tamaño del conjunto de datos que normalmente encontramos es entre los dos conjuntos de datos.

Supongamos que queremos reconocer diferentes tipos de sillas a partir de imágenes, y luego recomendar enlaces de compra a los usuarios. Un método posible es primero identificar 100 sillas comunes, tomar 1000 imágenes de diferentes ángulos para cada silla, y luego entrenar un modelo de clasificación en el conjunto de datos de imagen recopilada. Aunque este conjunto de datos de silla puede ser más grande que el conjunto de datos de moda-MNIST, el número de ejemplos es todavía menor que una décima parte de eso en ImageNet. Esto puede conducir a la adaptación excesiva de modelos complicados que son adecuados para ImageNet en este conjunto de datos de silla. Además, debido a la cantidad limitada de ejemplos de entrenamiento, la precisión del modelo entrenado puede no cumplir con los requisitos prácticos.

Con el fin de abordar los problemas anteriores, una solución obvia es recoger más datos. Sin embargo, recopilar y etiquetar datos puede tomar mucho tiempo y dinero. Por ejemplo, con el fin de recoger el conjunto de datos de ImageNet, los investigadores han gastado millones de dólares de la financiación de la investigación. Aunque el costo actual de recopilación de datos se ha reducido significativamente, este costo todavía no puede ser ignorado.

Otra solución es aplicar *transferir el aprendizaje* para transferir los conocimientos aprendidos del *conjunto de datos de origen* al *conjunto de datos de destino*. Por ejemplo, aunque la mayoría de las imágenes del conjunto de datos de ImageNet no tienen nada que ver con las sillas, el modelo formado en este conjunto de datos puede extraer características de imagen más generales, que pueden ayudar a identificar bordes, texturas, formas y composición de objetos.

## Pasos
En esta sección, introduciremos una técnica común en el aprendizaje de la transferencia: *final-tuning*. Como se muestra en [Referencia fig_finetune](https://d2l.ai/chapter_computer-vision/fine-tuning.html#fig-finetune), el ajuste consiste en los siguientes cuatro pasos:

1. Preentren un modelo de red neuronal, es decir, el modelo *fuente*, en un conjunto de datos de origen (por ejemplo, el conjunto de datos ImageNet).
1. Crear un nuevo modelo de red neuronal, es decir, el *modelo de destino*. Esto copia todos los diseños de modelos y sus parámetros en el modelo de origen excepto la capa de salida. Asumimos que estos parámetros de modelo contienen el conocimiento aprendido del conjunto de datos de origen y este conocimiento también será aplicable al conjunto de datos de destino. También suponemos que la capa de salida del modelo de origen está estrechamente relacionada con las etiquetas del conjunto de datos de origen; por lo tanto, no se utiliza en el modelo de destino.
1. Añadir una capa de salida al modelo de destino, cuyo número de salidas es el número de categorías en el conjunto de datos de destino. Luego inicializar al azar los parámetros de modelo de esta capa.
1. Capacitar el modelo objetivo en el conjunto de datos objetivo, como un conjunto de datos de la silla. La capa de salida se entrenará desde cero, mientras que los parámetros de todas las demás capas se afinan en función de los parámetros del modelo de origen.

![Ajuste fino.](../recursos/originales/finetune.svg)
<a id="fig_finetune"></a>

Cuando los conjuntos de datos de destino son mucho más pequeños que los conjuntos de datos de origen, el ajuste fino ayuda a mejorar la capacidad de generalización de los modelos.

## Reconocimiento de perritos calientes
Demostremos un ajuste fino a través de un caso concreto: reconocimiento de hot dog. Afinaremos un modelo de ResNet en un pequeño conjunto de datos, que fue entrenado previamente en el conjunto de datos ImageNet. Este pequeño conjunto de datos consta de miles de imágenes con y sin hot dogs. Usaremos el modelo afinado para reconocer hot dogs a partir de imágenes.


In [ ]:
%matplotlib inline
import os
import torch
import torchvision
from torch import nn
from laboratorio import d2l

### Leyendo el conjunto de datos
** El conjunto de datos de perritos calientes que usamos fue tomado de imágenes en línea**. Este conjunto de datos consta de 1400 imágenes de clase positiva que contienen perritos calientes, y como muchas imágenes de clase negativa que contienen otros alimentos. 1000 imágenes de ambas clases se utilizan para el entrenamiento y el resto son para pruebas.

Después de descomprimir el conjunto de datos descargado, obtenemos dos carpetas `hotdog/train` y `hotdog/test`. Ambas carpetas tienen subcarpetas `hotdog` y `not-hotdog`, cualquiera de las cuales contiene imágenes de la clase correspondiente.


In [ ]:
#@save
d2l.DATA_HUB['hotdog'] = (d2l.DATA_URL + 'hotdog.zip',
                         'fba480ffa8aa7e0febbb511d181409f899b9baa5')

data_dir = d2l.download_extract('hotdog')

Creamos dos instancias para leer todos los archivos de imagen en los conjuntos de datos de entrenamiento y pruebas, respectivamente.


In [ ]:
train_imgs = torchvision.datasets.ImageFolder(os.path.join(data_dir, 'train'))
test_imgs = torchvision.datasets.ImageFolder(os.path.join(data_dir, 'test'))

Los primeros 8 ejemplos positivos y las últimas 8 imágenes negativas se muestran a continuación. Como puede ver, **las imágenes varían en tamaño y proporción de aspecto**.


In [ ]:
hotdogs = [train_imgs[i][0] for i in range(8)]
not_hotdogs = [train_imgs[-i - 1][0] for i in range(8)]
d2l.show_images(hotdogs + not_hotdogs, 2, 8, scale=1.4);

Durante el entrenamiento, primero recortamos un área aleatoria de tamaño aleatorio y la relación de aspecto aleatorio de la imagen, y luego escalamos este área a una imagen de entrada $224 \times 224$. Durante la prueba, escalamos tanto la altura y el ancho de una imagen a 256 píxeles, y luego recortamos un área central $224 \times 224$ como entrada. Además, para los tres canales de color RGB (rojo, verde y azul) *estandarizamos* sus valores canal por canal. Concretamente, el valor medio de un canal se resta de cada valor de ese canal y luego el resultado se divide por la desviación estándar de ese canal.

[Ampliaciones de datos]


In [ ]:
# Especifique las medias y las desviaciones estándar de los tres canales RGB a
# estandarizar cada canal
normalize = torchvision.transforms.Normalize(
    [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_augs = torchvision.transforms.Compose([
    torchvision.transforms.RandomResizedCrop(224),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.ToTensor(),
    normalize])

test_augs = torchvision.transforms.Compose([
    torchvision.transforms.Resize([256, 256]),
    torchvision.transforms.CenterCrop(224),
    torchvision.transforms.ToTensor(),
    normalize])

### Definición e inicialización del modelo

Utilizamos ResNet-18, que estaba preentrenado en el conjunto de datos de ImageNet, como modelo de origen. Aquí especificamos `pretrained=True` para descargar automáticamente los parámetros del modelo preentrenado. Si este modelo se utiliza por primera vez, se requiere conexión a Internet para su descarga.


In [ ]:
pretrained_net = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)

La instancia de modelo de fuente preentrenada contiene una serie de capas de características y una capa de salida `fc`. El propósito principal de esta división es facilitar el ajuste de los parámetros de modelo de todas las capas excepto la capa de salida. La variable miembro `fc` del modelo de fuente se indica a continuación.


### Nota docente de Hespérides

Comprueba primero las formas y el supuesto arquitectónico: localidad, compartición de pesos o conexión residual. El explorador permite seguir ventana, multiplicaciones y suma. En PyTorch, Conv2d implementa correlación cruzada; en aprendizaje profundo se suele llamar convolución a esta operación. El autoencoder 91 amplía el patrón MLP con reconstrucción; VAE se trata como contraste conceptual, al no existir un original válido en las fuentes locales.

Vínculo con los apuntes: sesión 4, «Transferencia».


In [ ]:
pretrained_net.fc

Como una capa totalmente conectada, transforma las salidas finales de la agrupación global media de ResNet en 1000 salidas de clase del conjunto de datos ImageNet. A continuación, construimos una nueva red neuronal como modelo de destino. Se define de la misma manera que el modelo de fuente preentrenada, excepto que su número de salidas en la capa final se establece en el número de clases en el conjunto de datos objetivo (en lugar de 1000).

En el siguiente código, los parámetros del modelo antes de la capa de salida de la instancia del modelo de destino `finetune_net` Como estos parámetros de modelo se obtuvieron mediante preentrenamiento en ImageNet, son efectivos. Por lo tanto, sólo podemos usar una pequeña tasa de aprendizaje para *fina-tune* tales parámetros preentrenados. En contraste, los parámetros de modelo en la capa de salida se inicializan aleatoriamente y generalmente requieren una mayor tasa de aprendizaje para ser aprendidos desde cero. $\eta$, una tasa de aprendizaje de $10\eta$ se utilizará para iterar los parámetros del modelo en la capa de salida.


In [ ]:
finetune_net = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
finetune_net.fc = nn.Linear(finetune_net.fc.in_features, 2)
nn.init.xavier_uniform_(finetune_net.fc.weight);

### Fine-tuning del modelo

Primero, definimos una función de entrenamiento `train_fine_tuning` que utiliza ajuste fino para que se pueda llamar varias veces.


In [ ]:
# Si `param_group=True`, los parámetros del modelo en la capa de salida será
# actualizado utilizando una tasa de aprendizaje diez veces mayor
def train_fine_tuning(net, learning_rate, batch_size=128, num_epochs=5,
                      param_group=True):
    train_iter = torch.utils.data.DataLoader(torchvision.datasets.ImageFolder(
        os.path.join(data_dir, 'train'), transform=train_augs),
        batch_size=batch_size, shuffle=True)
    test_iter = torch.utils.data.DataLoader(torchvision.datasets.ImageFolder(
        os.path.join(data_dir, 'test'), transform=test_augs),
        batch_size=batch_size)
    devices = d2l.try_all_gpus()
    loss = nn.CrossEntropyLoss(reduction="none")
    if param_group:
        params_1x = [param for name, param in net.named_parameters()
             if name not in ["fc.weight", "fc.bias"]]
        trainer = torch.optim.SGD([{'params': params_1x},
                                   {'params': net.fc.parameters(),
                                    'lr': learning_rate * 10}],
                                lr=learning_rate, weight_decay=0.001)
    else:
        trainer = torch.optim.SGD(net.parameters(), lr=learning_rate,
                                  weight_decay=0.001)
    d2l.train_ch13(net, train_iter, test_iter, loss, trainer, num_epochs,
                   devices)

**Ajustamos la tasa de aprendizaje base a un pequeño valor** para *finar-afinar* los parámetros del modelo obtenidos mediante preentrenamiento. Basados en los ajustes anteriores, entrenaremos los parámetros de la capa de salida del modelo objetivo desde cero usando una tasa de aprendizaje diez veces mayor.


In [ ]:
train_fine_tuning(finetune_net, 5e-5)

**Para comparar,** definimos un modelo idéntico, pero **inicializamos todos sus parámetros de modelo a valores aleatorios**. Ya que todo el modelo necesita ser entrenado desde cero, podemos utilizar una mayor tasa de aprendizaje.


In [ ]:
scratch_net = torchvision.models.resnet18()
scratch_net.fc = nn.Linear(scratch_net.fc.in_features, 2)
train_fine_tuning(scratch_net, 5e-4, param_group=False)

Como podemos ver, el modelo afinado tiende a funcionar mejor para la misma época porque sus valores de parámetros iniciales son más efectivos.

## Resumen
* Transferir el aprendizaje transfiere el conocimiento aprendido del conjunto de datos de origen al conjunto de datos de destino. El ajuste es una técnica común para transferir el aprendizaje.
* El modelo de destino copia todos los diseños de modelo con sus parámetros del modelo de origen, excepto la capa de salida, y afina estos parámetros basados en el conjunto de datos de destino. En contraste, la capa de salida del modelo de destino necesita ser entrenada desde cero.
* Por lo general, los parámetros de ajuste utilizan una tasa de aprendizaje menor, mientras que el entrenamiento de la capa de salida desde cero puede utilizar una mayor tasa de aprendizaje.

## Ejercicios
1. Siga aumentando la tasa de aprendizaje de `finetune_net`. ¿Cómo cambia la precisión del modelo?
2. Ajuste más adelante los hiperparametros de `finetune_net` y `scratch_net` en el experimento comparativo. ¿Todavía difieren en precisión?
3. Establecer los parámetros antes de la capa de salida de `finetune_net` a los del modelo de origen y *no * actualizarlos durante el entrenamiento. ¿Cómo cambia la precisión del modelo? Puede utilizar el siguiente código.


In [ ]:
for param in finetune_net.parameters():
    param.requires_grad = False

4. De hecho, hay una clase "hotdog" en el conjunto de datos `ImageNet`. Su parámetro de peso correspondiente en la capa de salida se puede obtener a través del siguiente código. ¿Cómo podemos aprovechar este parámetro de peso?


In [ ]:
weight = pretrained_net.fc.weight
hotdog_w = torch.split(weight.data, 1, dim=0)[934]
hotdog_w.shape

[Debate del original](https://discuss.d2l.ai/t/1439)
